Cargar

In [1]:
import json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from redes_comun import Atencion, fijar_semilla, entrenar, predecir  

CARPETA = "../data/e3"
df = pd.read_json(f"{CARPETA}/datos_E3.jsonl", lines=True)
df["fila"] = np.arange(len(df))

with open(f"{CARPETA}/clases_tipo.json", encoding="utf-8") as f:
    clases_tipo = json.load(f)
clase_a_num = {c: i for i, c in enumerate(clases_tipo)}

with open(f"{CARPETA}/vocabulario_secuencias.json", encoding="utf-8") as f:
    vocab = json.load(f)                       
matriz_eswiki = np.load(f"{CARPETA}/matriz_eswiki.npy")
print(len(vocab), matriz_eswiki.shape)          

18437 (18437, 300)


In [2]:
MAX_LEN = 150   #valor fijado 01_datos

def a_indices(tokens):
    ids = [vocab.get(t, vocab.get(t.lower(), 1)) for t in tokens[:MAX_LEN]]
    return ids + [0] * (MAX_LEN - len(ids))

X_todo = torch.tensor([a_indices(t) for t in df["tokens"]])
print(X_todo.shape)                             # (7843, MAX_LEN)

tipo = df[df["etiqueta_tipo"].notna()]
tipo_train = tipo[tipo.split == "train"].explode("etiqueta_tipo")
tipo_dev   = tipo[tipo.split == "dev"]
tipo_test  = tipo[tipo.split == "test"]
tipo_dev  = tipo_dev.assign(etiqueta_tipo=tipo_dev["etiqueta_tipo"].str[0])
tipo_test = tipo_test.assign(etiqueta_tipo=tipo_test["etiqueta_tipo"].str[0])

# dev/test solo Political
dev_pol  = tipo_dev[tipo_dev["corpus"] == "political"]
test_pol = tipo_test[tipo_test["corpus"] == "political"]

def tensores(d):
    X = X_todo[d["fila"].values]
    y = torch.tensor(d["etiqueta_tipo"].map(clase_a_num).values)
    return X, y

X_tr, y_tr = tensores(tipo_train)
X_dv, y_dv = tensores(dev_pol)
X_te, y_te = tensores(test_pol)
print(X_tr.shape, X_dv.shape, X_te.shape)

torch.Size([7843, 150])
torch.Size([3200, 150]) torch.Size([289, 150]) torch.Size([344, 150])


C:\Users\alvarodela.herran\AppData\Local\Temp\ipykernel_28508\2518181364.py:22: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:214.)
  X = X_todo[d["fila"].values]


Red

In [3]:
class BiLSTMAtencion(nn.Module):
    def __init__(self, matriz, n_clases, oculto, dropout, congelar=True, aleatorio=False):
        super().__init__()
        n_vocab, dim = matriz.shape
        if aleatorio:   # variante "aleatorio": mismo tamaño, sin vectores de eswiki
            self.embedding = nn.Embedding(n_vocab, dim, padding_idx=0)
        else:           # congelar=True: eswiki fijo; congelar=False: se ajusta al entrenar
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(matriz, dtype=torch.float32), freeze=congelar, padding_idx=0)
        self.lstm = nn.LSTM(dim, oculto, bidirectional=True, batch_first=True)
        self.atencion = Atencion(2 * oculto)
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(2 * oculto, n_clases)

    def forward(self, x):
        mascara = x != 0                         
        H, _ = self.lstm(self.embedding(x))     
        vector, pesos = self.atencion(H, mascara)
        return self.linear(self.dropout(vector)), pesos   

Pre optuna

In [4]:
t0 = time.time()
fijar_semilla(0)                                  # siempre antes de crear la red
red = BiLSTMAtencion(matriz_eswiki, n_clases=11, oculto=64, dropout=0.3)
f1 = entrenar(red, X_tr, y_tr, X_dv, y_dv, lr=1e-3, batch_size=64,
              n_clases=11, max_epocas=2)
print(f"F1 macro dev Political: {f1:.3f}  ({time.time()-t0:.0f} s)")

F1 macro dev Political: 0.203  (13 s)


Optuna

In [5]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo(trial):
    lr         = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout    = trial.suggest_float("dropout", 0.1, 0.6)
    oculto     = trial.suggest_categorical("oculto", [32, 64, 128])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    fijar_semilla(0)
    red = BiLSTMAtencion(matriz_eswiki, n_clases=11, oculto=oculto, dropout=dropout)
    return entrenar(red, X_tr, y_tr, X_dv, y_dv, lr=lr, batch_size=batch_size,
                    n_clases=11, trial=trial)

def optimizar_lstm(n_trials=15):
    t0 = time.time()
    estudio = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )
    estudio.optimize(objetivo, n_trials=n_trials, show_progress_bar=True)
    print(f"F1 macro dev Political = {estudio.best_value:.3f} ({(time.time()-t0)/60:.1f} min)")
    print(estudio.best_params)
    with open("mejores_params_bilstm_tipo.json", "w") as f:
        json.dump(estudio.best_params, f, indent=2)
    return estudio.best_params

In [7]:
optimizar_lstm(n_trials=20)

  0%|          | 0/20 [00:00<?, ?it/s]

F1 macro dev Political = 0.354 (20.6 min)
{'lr': 0.003796611592406361, 'dropout': 0.34787630263804636, 'oculto': 128, 'batch_size': 64}


{'lr': 0.003796611592406361,
 'dropout': 0.34787630263804636,
 'oculto': 128,
 'batch_size': 64}

In [10]:
import os
def entrenar_variante_lstm(nombre, train_df, congelar=True, aleatorio=False, semillas=(0, 1, 2)):
    with open("mejores_params_bilstm_tipo.json") as f:
        p = json.load(f)
    X_a, y_a = tensores(train_df)
    print(nombre, "ejemplos por clase:", torch.bincount(y_a, minlength=11).tolist())

    f1s = []
    for s in semillas:
        t0 = time.time()
        fijar_semilla(s)                       
        red = BiLSTMAtencion(matriz_eswiki, n_clases=11, oculto=p["oculto"],
                             dropout=p["dropout"], congelar=congelar, aleatorio=aleatorio)
        f1 = entrenar(red, X_a, y_a, X_dv, y_dv, lr=p["lr"],
                      batch_size=p["batch_size"], n_clases=11)
        f1s.append(f1)

        pred_dv, pred_te = predecir(red, X_dv), predecir(red, X_te)
        salida = pd.concat([
            pd.DataFrame({"id": dev_pol["id"].values, "split": "dev",
                          "real": [clases_tipo[i] for i in y_dv.numpy()],
                          "predicho": [clases_tipo[i] for i in pred_dv]}),
            pd.DataFrame({"id": test_pol["id"].values, "split": "test",
                          "real": [clases_tipo[i] for i in y_te.numpy()],
                          "predicho": [clases_tipo[i] for i in pred_te]}),
        ])
        os.makedirs("predicciones", exist_ok=True)
        salida.to_csv(f"predicciones/tipo__bilstm__{nombre}__s{s}.csv", index=False)
        print(f"  semilla {s}: F1 = {f1:.3f} ({time.time()-t0:.0f} s)")

    print(f"{nombre}: F1 macro dev Political = {np.mean(f1s):.3f} ± {np.std(f1s):.3f}  {np.round(f1s, 3)}")

In [11]:
entrenar_variante_lstm("eswiki_congelado", tipo_train)

eswiki_congelado ejemplos por clase: [631, 202, 319, 174, 106, 273, 337, 343, 342, 404, 69]
  semilla 0: F1 = 0.354 (138 s)
  semilla 1: F1 = 0.349 (110 s)
  semilla 2: F1 = 0.347 (145 s)
eswiki_congelado: F1 macro dev Political = 0.350 ± 0.003  [0.354 0.349 0.347]


In [12]:
# E2: embeddings
entrenar_variante_lstm("eswiki_ajustado", tipo_train, congelar=False)
entrenar_variante_lstm("aleatorio", tipo_train, aleatorio=True)

# E3: datos de entrenamiento
entrenar_variante_lstm("solo_political", tipo_train[tipo_train["corpus"] == "political"])

eswiki_ajustado ejemplos por clase: [631, 202, 319, 174, 106, 273, 337, 343, 342, 404, 69]
  semilla 0: F1 = 0.317 (96 s)
  semilla 1: F1 = 0.329 (67 s)
  semilla 2: F1 = 0.333 (213 s)
eswiki_ajustado: F1 macro dev Political = 0.326 ± 0.007  [0.317 0.329 0.333]
aleatorio ejemplos por clase: [631, 202, 319, 174, 106, 273, 337, 343, 342, 404, 69]
  semilla 0: F1 = 0.313 (70 s)
  semilla 1: F1 = 0.294 (84 s)
  semilla 2: F1 = 0.290 (81 s)
aleatorio: F1 macro dev Political = 0.299 ± 0.010  [0.313 0.294 0.29 ]
solo_political ejemplos por clase: [308, 30, 194, 46, 12, 65, 208, 134, 34, 225, 69]
  semilla 0: F1 = 0.259 (35 s)
  semilla 1: F1 = 0.305 (64 s)
  semilla 2: F1 = 0.304 (57 s)
solo_political: F1 macro dev Political = 0.289 ± 0.022  [0.259 0.305 0.304]
